# Поиск релевантных объявлений на Авито

**Задача:** для каждого поискового запроса вернуть 50 наиболее релевантных объявлений из каталога.

**Метрика:** Recall@50 — доля правильных объявлений, попавших в топ-50 предсказаний, усреднённая по запросам.

**Финальный результат (тестовый набор): 0.7745**

**Архитектура решения:**

```
Запрос ──┬── BM25 (лексический поток) ──────── top-200 ──┐
         │                                               ├── Weighted RRF ── top-50
         └── Bi-Encoder ruBERT (dense поток) ── top-200 ──┘
```

Два независимых потока извлекают кандидатов из каталога: лексический (BM25) и нейросетевой (Bi-Encoder). Их результаты объединяются через Reciprocal Rank Fusion. Каждый поток применяет двухуровневый фильтр по городу и категории.


In [ ]:
!pip install -q bm25s PyStemmer sentence-transformers


## 1. Данные и признаки

Исходные данные:
- `train.parquet` — обучающая выборка (пары запрос–объявление с метаданными)
- `benchmark_items.parquet` — тестовый каталог объявлений
- `benchmark_queries.parquet` — тестовые запросы

Используемые признаки:

| Признак | Тип | Назначение |
|---|---|---|
| `item_title_raw` | текст | Основной семантический сигнал для BM25 и Bi-Encoder |
| `item_infm_params_text` | текст | Характеристики товара, дополняющие заголовок |
| `item_description_raw` | текст | Описание объявления (слабый текстовый сигнал, используется в BM25) |
| `item_location_id` | int | Фильтрация кандидатов по городу поиска |
| `item_category_id` | int | Фильтрация по категории запроса |
| `search_is_delivery_search` | int | Флаг доставки: если 1, фильтрация по городу не применяется |

Текстовые поля покрывают семантику объявления (что продаётся и с какими характеристиками). Геолокация критична: пользователь, как правило, ищет товары в своём городе. Категория также участвует в фильтрации кандидатов, хотя в данном датасете её вклад ограничен (подробнее — в разделе 6). Флаг доставки определяет, нужна ли привязка к городу.

**Масштаб данных:** `train.parquet` содержит 497 673 строки, 74 529 уникальных запросов, 344 825 уникальных объявлений и 445 261 уникальную пару (запрос, объявление) — то есть исходный файл содержит около 52 тыс. строк-дублей одной и той же пары (запрос, объявление), не несущих дополнительной информации для recall-задачи.



In [ ]:
import os
import gc
import re
import random
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from tqdm.auto import tqdm

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

# Примечание про воспроизводимость: seed фиксирует random/numpy/torch и включает
# детерминированный режим cuDNN. Обучение Bi-Encoder идёт в FP16 (Mixed Precision) —
# это ускоряет обучение, но не даёт 100% побитовой детерминированности на GPU: при
# повторном запуске Recall@50 может отличаться в пределах ~0.001-0.002 от заявленного.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Автоматический поиск директории с данными (Kaggle, ../data, data, .)
DATA_DIR = '.'
for candidate in ['/kaggle/input/datasets/bogdanborovoy/avito-ds', '../data', 'data', '.']:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, 'train.parquet')):
        DATA_DIR = candidate
        break
print(f"Data directory: {DATA_DIR}")
train_path = os.path.join(DATA_DIR, 'train.parquet')
bench_items_path = os.path.join(DATA_DIR, 'benchmark_items.parquet')
bench_queries_path = os.path.join(DATA_DIR, 'benchmark_queries.parquet')

# Загружаем только нужные колонки для экономии памяти
train_cols = [
    'search_query', 'item_id', 'search_location_id', 'search_category',
    'search_is_delivery_search', 'item_title_raw', 'item_infm_params_text',
    'item_description_raw', 'item_location_id', 'item_category_id'
]
df_train = pd.read_parquet(train_path, columns=train_cols)

print(f"train.parquet: {len(df_train):,} строк")
print(f"  Уникальных запросов: {df_train['search_query'].nunique():,}")
print(f"  Уникальных объявлений: {df_train['item_id'].nunique():,}")
print(f"  Уникальных пар (запрос, объявление): {df_train.drop_duplicates(['search_query','item_id']).shape[0]:,}")


## 2. Предобработка и методология оценки

**Нормализация текста:** приведение к нижнему регистру, удаление спецсимволов, нормализация пробелов. Минимальная обработка — модели (BM25 со стеммером и BERT с субтокенизацией) справляются с вариативностью форм.

**Валидация:** 80/20 сплит по уникальным поисковым запросам. Это предотвращает утечку данных — один и тот же запрос не может попасть одновременно в обучающую и валидационную выборки.


In [ ]:
def clean_text(text: str) -> str:
    """Нормализация текста: lowercase, удаление спецсимволов, нормализация пробелов."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zа-я0-9\s]', ' ', text)
    return ' '.join(text.split())

def evaluate_recall_at_k(predictions: dict, ground_truth: dict, k: int = 50) -> float:
    """Recall@K: доля правильных ответов в топ-K предсказаний, усреднённая по запросам."""
    recalls = []
    for query, true_items in ground_truth.items():
        if not true_items:
            continue
        top_k = predictions.get(query, [])[:k]
        hits = len(set(top_k) & set(true_items))
        recalls.append(hits / len(true_items))
    return float(np.mean(recalls)) if recalls else 0.0

# --- Сплит 80/20 по уникальным запросам ---
unique_queries = df_train['search_query'].unique()
np.random.seed(42)
shuffled_q = np.random.permutation(unique_queries)
split_idx = int(len(shuffled_q) * 0.8)

train_q_set = set(shuffled_q[:split_idx])
val_q_set = set(shuffled_q[split_idx:])

train_df = df_train[df_train['search_query'].isin(train_q_set)]
val_df = df_train[df_train['search_query'].isin(val_q_set)]

val_queries_list = list(val_df['search_query'].unique())
val_ground_truth = val_df.groupby('search_query')['item_id'].apply(set).to_dict()

print(f"Обучающих запросов: {len(train_q_set):,}")
print(f"Валидационных запросов: {len(val_queries_list):,}")

# --- Корпус объявлений ---
items_corpus = df_train.drop_duplicates(subset=['item_id']).copy()
corpus_ids = items_corpus['item_id'].tolist()
corpus_ids_arr = np.array(corpus_ids)
print(f"Уникальных объявлений в корпусе: {len(items_corpus):,}")

# --- Маппинги для фильтрации ---
query_locs_map = val_df.groupby('search_query')['search_location_id'].apply(set).to_dict()
query_cats_map = val_df.groupby('search_query')['search_category'].apply(set).to_dict()
query_delivery_map = val_df.groupby('search_query')['search_is_delivery_search'].max().to_dict()

# --- Тензоры метаданных корпуса для GPU-фильтрации ---
loc_tensor = torch.tensor(items_corpus['item_location_id'].fillna(0).to_numpy(dtype=np.int32), dtype=torch.int32, device=device)
cat_tensor = torch.tensor(items_corpus['item_category_id'].fillna(0).to_numpy(dtype=np.int32), dtype=torch.int32, device=device)

# --- Предварительная нормализация текстов ---
title_clean = items_corpus['item_title_raw'].apply(clean_text)
params_clean = items_corpus['item_infm_params_text'].apply(clean_text)
desc_clean = items_corpus['item_description_raw'].fillna('').str[:600].apply(clean_text)


## 3. Лексический поиск (BM25)

BM25 — стандартный алгоритм информационного поиска, основанный на TF-IDF с нормализацией по длине документа.

**Ключевые решения:**

1. **Обогащение текста документа:** заголовок повторяется 3 раза, затем конкатенируются параметры и описание (обрезанное до 600 символов). Утроение заголовка повышает вес наиболее информативного поля.

2. **Стемминг:** русский стеммер PyStemmer для нормализации словоформ ("купить" → "куп").

3. **Параметры BM25:** k1=1.5 (стандарт), b=0.4 (ниже стандартного 0.75). Пониженный b обоснован тем, что заголовки на Авито короткие и однородные по длине — сильная нормализация по длине документа не нужна.

4. **Двухуровневый ярусный фильтр:** из топ-5000 BM25-кандидатов отбираются 200 лучших с учётом приоритетов:
   - Для обычных запросов: (город + категория) > (только категория) > остальные
   - Для запросов с доставкой: только по категории


In [ ]:
import bm25s
import Stemmer

# Обогащённые тексты: Title x3 + Params + Description
corpus_bm25_texts = (
    title_clean + " " + title_clean + " " + title_clean + " " +
    params_clean + " " +
    desc_clean
).tolist()

stemmer = Stemmer.Stemmer('russian')

print("Токенизация корпуса со стеммингом...")
corpus_bm25_tokens = bm25s.tokenize(corpus_bm25_texts, stopwords='russian', stemmer=stemmer)

val_queries_clean = [clean_text(q) for q in val_queries_list]
val_bm25_tokens = bm25s.tokenize(val_queries_clean, stopwords='russian', stemmer=stemmer)

print("Построение индекса BM25 (k1=1.5, b=0.4)...")
retriever_v2 = bm25s.BM25(k1=1.5, b=0.4)
retriever_v2.index(corpus_bm25_tokens)

print("Извлечение top-5000 кандидатов BM25...")
v2_results_max, _ = retriever_v2.retrieve(val_bm25_tokens, k=5000)

# Ярусный фильтр: город+категория -> категория -> остальные
num_val_queries = len(val_queries_list)
k_bm25_cand = v2_results_max.shape[1]
top_bm25_keep = 200
bm25_preds_top = {}
rank_penalty = torch.arange(k_bm25_cand, device=device, dtype=torch.int32)
batch_size_gpu = 2000

for start_idx in range(0, num_val_queries, batch_size_gpu):
    end_idx = min(start_idx + batch_size_gpu, num_val_queries)
    batch_queries = val_queries_list[start_idx:end_idx]
    batch_cand_indices = torch.tensor(v2_results_max[start_idx:end_idx, :k_bm25_cand], dtype=torch.long, device=device)
    batch_locs = loc_tensor[batch_cand_indices]
    batch_cats = cat_tensor[batch_cand_indices]

    for b_i, q_text in enumerate(batch_queries):
        allowed_locs = torch.tensor(list(query_locs_map.get(q_text, set())), device=device, dtype=torch.int32)
        allowed_cats = torch.tensor(list(query_cats_map.get(q_text, set())), device=device, dtype=torch.int32)
        is_delivery = query_delivery_map.get(q_text, 0)

        match_loc = torch.isin(batch_locs[b_i], allowed_locs)
        match_cat = torch.isin(batch_cats[b_i], allowed_cats)

        if is_delivery == 1:
            priority = match_cat.int() * 20000000 - rank_penalty
        else:
            priority = (match_loc & match_cat).int() * 20000000 + match_cat.int() * 10000000 - rank_penalty

        top_idx = torch.topk(priority, k=top_bm25_keep).indices
        p_idx = batch_cand_indices[b_i, top_idx].cpu().numpy()
        bm25_preds_top[q_text] = corpus_ids_arr[p_idx].tolist()

bm25_only_preds = {q: items[:50] for q, items in bm25_preds_top.items()}
bm25_recall = evaluate_recall_at_k(bm25_only_preds, val_ground_truth, k=50)
print(f"BM25 v2 Recall@50: {bm25_recall:.4f}")


## 4. Нейросетевой поиск (Bi-Encoder)

**Модель:** `cointegrated/rubert-tiny2` — компактная русскоязычная модель (29M параметров, 3 transformer-слоя, размерность эмбеддингов 312). Выбрана из-за ограничений GPU-памяти на Kaggle (Tesla T4, ~15 ГБ): нужно хранить эмбеддинги всего корпуса в памяти.

**Архитектура:** Bi-Encoder с CLS-пулингом. Запрос и документ кодируются независимо, релевантность определяется косинусным сходством. Это позволяет предвычислить эмбеддинги корпуса один раз.

**Обучение:**
- 300 000 пар (запрос, заголовок + параметры) из обучающей выборки
- Функция потерь: `MultipleNegativesRankingLoss` — контрастивный лосс, использующий остальные элементы батча как негативные примеры (in-batch negatives)
- 4 эпохи, batch_size=128, lr=2e-5, warmup=400 шагов, weight_decay=0.01
- Mixed Precision (FP16) для ускорения и экономии памяти


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses, models
from torch.utils.data import DataLoader

# Очистка памяти перед обучением
del corpus_bm25_texts, corpus_bm25_tokens, val_bm25_tokens
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Подготовка обучающих пар
cols_pairs = ['search_query', 'item_title_raw', 'item_infm_params_text']
df_pairs = train_df[cols_pairs].dropna(subset=['search_query', 'item_title_raw']).drop_duplicates(subset=['search_query', 'item_title_raw'])

MAX_PAIRS = 300000
if len(df_pairs) > MAX_PAIRS:
    df_pairs = df_pairs.sample(MAX_PAIRS, random_state=42)

queries_p = df_pairs['search_query'].tolist()
titles_p = df_pairs['item_title_raw'].tolist()
params_p = df_pairs['item_infm_params_text'].fillna('').str[:150].tolist()
del df_pairs
gc.collect()

train_examples = []
for q, t, p in zip(queries_p, titles_p, params_p):
    cq = clean_text(q)
    ct = clean_text(t)
    if p:
        cp = clean_text(p)
        if cp:
            ct = ct + " " + cp
    if cq and ct:
        train_examples.append(InputExample(texts=[cq, ct]))

del queries_p, titles_p, params_p
gc.collect()
print(f"Обучающих пар: {len(train_examples):,}")

# Инициализация модели с CLS-пулингом
word_embedding_model = models.Transformer('cointegrated/rubert-tiny2', max_seq_length=64)
pooling_model = models.Pooling(
    word_embedding_dimension=word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=False,
    pooling_mode_cls_token=True
)
ft_model = SentenceTransformer(modules=[word_embedding_model, pooling_model], device=device)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=128)
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

print("Обучение: 4 эпохи, lr=2e-5, warmup=400, FP16...")
ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=400,
    optimizer_params={'lr': 2e-5},
    weight_decay=0.01,
    use_amp=torch.cuda.is_available(),
    show_progress_bar=True
)

ft_model.save('./best_rubert_model')
print("Модель сохранена в ./best_rubert_model")

del train_dataloader, train_loss, train_examples
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. Объединение потоков (Weighted RRF)

**Reciprocal Rank Fusion** объединяет ранжированные списки из разных источников без обучения. Каждому элементу присваивается оценка, обратно пропорциональная его рангу:

$$\text{score}(item) = \frac{w_{bm25}}{k + rank_{bm25}} + \frac{w_{dense}}{k + rank_{dense}}$$

**Параметры** (подобраны на валидационной выборке):
- $k = 20$ — сглаживающий параметр
- $w_{bm25} = 1.0$ — вес лексического потока
- $w_{dense} = 0.9$ — вес нейросетевого потока

RRF выбран по нескольким причинам:
1. Не требует обучения (в отличие от, например, CatBoost re-ranking, который давал лишь +0.01 и был отклонён).
2. Робастен к различиям в шкалах оценок разных моделей.
3. Два потока дополняют друг друга: BM25 точен при совпадении терминов, Bi-Encoder — при семантическом сходстве.


In [ ]:
emb_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Кодирование корпуса и запросов через дообученный ruBERT
print("Кодирование корпуса (FP16)...")
corpus_rich_titles = (title_clean + " " + params_clean.str[:200]).str.strip().tolist()

with torch.no_grad():
    corpus_dense_embs = ft_model.encode(
        corpus_rich_titles, batch_size=1024, show_progress_bar=True,
        normalize_embeddings=True, convert_to_tensor=True, device=device
    ).to(dtype=emb_dtype)

    val_dense_embs = ft_model.encode(
        val_queries_clean, batch_size=1024, show_progress_bar=True,
        normalize_embeddings=True, convert_to_tensor=True, device=device
    ).to(dtype=emb_dtype)

del corpus_rich_titles
gc.collect()

# Извлечение Dense-кандидатов с фильтрацией
print("Извлечение Dense-кандидатов...")
top_dense_keep = 200
dense_preds_top = {}
chunk_size_dense = 500

for start_idx in tqdm(range(0, num_val_queries, chunk_size_dense), desc="Dense Search"):
    end_idx = min(start_idx + chunk_size_dense, num_val_queries)
    q_chunk = val_dense_embs[start_idx:end_idx]
    sims_chunk = torch.matmul(q_chunk, corpus_dense_embs.T)
    s_dtype = sims_chunk.dtype

    for b_i, q_text in enumerate(val_queries_list[start_idx:end_idx]):
        allowed_locs = list(query_locs_map.get(q_text, set()))
        allowed_cats = list(query_cats_map.get(q_text, set()))
        is_delivery = query_delivery_map.get(q_text, 0)

        match_loc = (loc_tensor == allowed_locs[0]) if len(allowed_locs) == 1 else torch.isin(loc_tensor, torch.tensor(allowed_locs, device=device, dtype=torch.int32))
        match_cat = (cat_tensor == allowed_cats[0]) if len(allowed_cats) == 1 else torch.isin(cat_tensor, torch.tensor(allowed_cats, device=device, dtype=torch.int32))

        if is_delivery == 1:
            priority = sims_chunk[b_i] + match_cat.to(dtype=s_dtype) * 20.0
        else:
            priority = sims_chunk[b_i] + (match_loc & match_cat).to(dtype=s_dtype) * 20.0 + match_cat.to(dtype=s_dtype) * 10.0

        top_d_idx = torch.topk(priority, k=top_dense_keep).indices
        dense_preds_top[q_text] = corpus_ids_arr[top_d_idx.cpu().numpy()].tolist()

# Оценка Dense-потока
dense_only_preds = {q: items[:50] for q, items in dense_preds_top.items()}
dense_recall = evaluate_recall_at_k(dense_only_preds, val_ground_truth, k=50)

# --- Weighted RRF ---
print("Слияние потоков через Weighted RRF...")
best_params = {'k_rrf': 20, 'w_bm25': 1.0, 'w_dense': 0.9}
opt_k = best_params['k_rrf']
opt_wb = best_params['w_bm25']
opt_wd = best_params['w_dense']

rrf_preds = {}
for q in val_queries_list:
    scores = defaultdict(float)
    for r, itm in enumerate(bm25_preds_top.get(q, []), 1):
        scores[itm] += opt_wb / (opt_k + r)
    for r, itm in enumerate(dense_preds_top.get(q, []), 1):
        scores[itm] += opt_wd / (opt_k + r)
    rrf_preds[q] = sorted(scores.items(), key=lambda x: x[1], reverse=True)

rrf_preds_50 = {q: [itm for itm, _ in items[:50]] for q, items in rrf_preds.items()}
rrf_recall = evaluate_recall_at_k(rrf_preds_50, val_ground_truth, k=50)

# Итоговое сравнение
print()
print("=" * 50)
print("Результаты на валидации (Recall@50):")
print(f"  BM25 v2:                {bm25_recall:.4f}")
print(f"  Bi-Encoder (ruBERT):    {dense_recall:.4f}")
print(f"  Weighted RRF (BM25+BE): {rrf_recall:.4f}")
print("=" * 50)


## 6. Анализ ошибок

Для понимания ограничений модели проведён анализ валидационных предсказаний: распределение recall по запросам, влияние кросс-городской релевантности, разбивка по категориям, вклад каждого потока (BM25 vs Bi-Encoder) и потери на этапе ярусного фильтра.


In [ ]:
# 1. Распределение Recall@50 по запросам
recalls_per_query = []
for q, gt in val_ground_truth.items():
    preds = rrf_preds_50.get(q, [])[:50]
    hits = len(set(preds) & gt)
    recalls_per_query.append(hits / len(gt))

recalls_arr = np.array(recalls_per_query)

print("Распределение Recall@50 по запросам:")
print(f"  Среднее:     {recalls_arr.mean():.4f}")
print(f"  Медиана:     {np.median(recalls_arr):.4f}")
print(f"  recall = 1.0: {(recalls_arr == 1.0).sum()} ({(recalls_arr == 1.0).mean():.1%})")
print(f"  recall = 0.0: {(recalls_arr == 0.0).sum()} ({(recalls_arr == 0.0).mean():.1%})")
print(f"  recall >= 0.5: {(recalls_arr >= 0.5).sum()} ({(recalls_arr >= 0.5).mean():.1%})")

# 2. Анализ кросс-городской релевантности
# Проверяем: какая доля ground truth объявлений находится в другом городе
item_id_to_loc = dict(zip(items_corpus['item_id'], items_corpus['item_location_id']))
item_id_to_cat = dict(zip(items_corpus['item_id'], items_corpus['item_category_id']))

same_city_cat = 0
diff_city_same_cat = 0
other = 0
total = 0

for q, gt_items in val_ground_truth.items():
    q_locs = query_locs_map.get(q, set())
    q_cats = query_cats_map.get(q, set())
    for item_id in gt_items:
        loc = item_id_to_loc.get(item_id)
        cat = item_id_to_cat.get(item_id)
        total += 1
        if loc in q_locs and cat in q_cats:
            same_city_cat += 1
        elif cat in q_cats:
            diff_city_same_cat += 1
        else:
            other += 1

print()
print(f"Кросс-городская релевантность ({total:,} пар в ground truth):")
print(f"  Город + категория совпадают: {same_city_cat:,} ({same_city_cat/total:.1%})")
print(f"  Только категория (другой город): {diff_city_same_cat:,} ({diff_city_same_cat/total:.1%})")
print(f"  Ни то, ни другое: {other:,} ({other/total:.1%})")
print()
print("Вывод: значительная доля релевантных объявлений находится в других городах.")
print("Ярусный фильтр приоритизирует локальные результаты, но не исключает")
print("кросс-городские полностью — это компромисс между precision и recall.")


**Распределение Recall@50 по запросам (на валидации, 14 906 запросов):**

| Метрика | Значение |
|---|---|
| Среднее (= общий Recall@50) | 0.7057 |
| Медиана | 1.0000 |
| Recall = 1.0 | 8 560 запросов (57.4%) |
| Recall = 0.0 | 2 422 запроса (16.2%) |
| Recall ≥ 0.5 | 11 127 запросов (74.6%) |

Более половины запросов (57.4%) получают полное покрытие релевантных объявлений в топ-50, но у 16.2% запросов recall нулевой — это основной резерв для роста метрики.

**Кросс-городская релевантность (89 503 пары в ground truth):**

| Совпадение | Кол-во | Доля |
|---|---|---|
| Город + категория | 78 524 | 87.7% |
| Только категория (другой город) | 10 973 | 12.3% |
| Ни то, ни другое | 6 | 0.0% |

Вывод: подавляющее большинство релевантных объявлений (87.7%) находится в том же городе, что и запрос — это подтверждает корректность приоритизации локальных результатов (оговорка про то, что этот процент фактически измеряет в основном совпадение по городу, а не по категории — см. ниже). При этом 12.3% релевантных объявлений теряются в других городах — это случай, когда ярусный фильтр может ограничивать recall при полностью жёсткой фильтрации, но эта доля явно не объясняет большую часть нулевых recall (16.2% запросов): 12.3% кросс-городских случаев в разы меньше 16.2% полностью провальных запросов. Значит, основная причина нулевого recall — не город, а что-то другое. Три анализа ниже проверяют эти гипотезы напрямую.


### Разбивка recall по категориям

Проверяем, какие категории объявлений дают наибольший вклад в ошибки (самый низкий recall), среди категорий с достаточным числом запросов для устойчивой оценки.


In [ ]:
from collections import defaultdict

cat_recalls = defaultdict(list)
for q, gt in val_ground_truth.items():
    preds = set(rrf_preds_50.get(q, [])[:50])
    cats = query_cats_map.get(q, set())
    cat_key = list(cats)[0] if cats else None
    cat_recalls[cat_key].append(len(preds & gt) / len(gt))

df_cat = pd.DataFrame([
    {'category': c, 'n_queries': len(r), 'mean_recall': np.mean(r)}
    for c, r in cat_recalls.items()
]).sort_values('mean_recall')

print("Худшие 10 категорий по recall (среди тех, где запросов >= 20):")
print(df_cat[df_cat['n_queries'] >= 20].head(10).to_string(index=False))
print()
print("Лучшие 10 категорий по recall (среди тех, где запросов >= 20):")
print(df_cat[df_cat['n_queries'] >= 20].sort_values('mean_recall', ascending=False).head(10).to_string(index=False))


### Вклад каждого потока в финальный результат

Проверяем, сколько уникальных релевантных объявлений находит каждый поток (BM25 и Bi-Encoder) в топ-50 по отдельности, до объединения через RRF — это объясняет, за счёт чего именно RRF даёт прирост с 0.6330/0.6580 до 0.7057.


In [ ]:
bm25_only_hits = dense_only_hits = both_hits = neither_hits = total_gt = 0
for q, gt in val_ground_truth.items():
    b_set = set(bm25_preds_top.get(q, [])[:50])
    d_set = set(dense_preds_top.get(q, [])[:50])
    for item in gt:
        total_gt += 1
        in_b, in_d = item in b_set, item in d_set
        if in_b and in_d:
            both_hits += 1
        elif in_b:
            bm25_only_hits += 1
        elif in_d:
            dense_only_hits += 1
        else:
            neither_hits += 1

print(f"Только BM25: {bm25_only_hits} ({bm25_only_hits/total_gt:.1%})")
print(f"Только Dense: {dense_only_hits} ({dense_only_hits/total_gt:.1%})")
print(f"Оба потока: {both_hits} ({both_hits/total_gt:.1%})")
print(f"Ни один (в топ-50 своего потока): {neither_hits} ({neither_hits/total_gt:.1%})")


### Потери на этапе ярусного фильтра (BM25)

Проверяем, сколько релевантных объявлений присутствовало в сыром топ-5000 BM25-кандидатов (до применения ярусного фильтра по городу/категории), но было отсеяно фильтром при сокращении до топ-200.


In [ ]:
query_to_idx = {q: i for i, q in enumerate(val_queries_list)}
present_in_raw = lost_by_filter = 0

for q, gt in val_ground_truth.items():
    idx = query_to_idx[q]
    raw_ids = set(corpus_ids_arr[v2_results_max[idx]].tolist())
    final_ids = set(bm25_only_preds.get(q, []))
    for item in gt:
        if item in raw_ids:
            present_in_raw += 1
            if item not in final_ids:
                lost_by_filter += 1

print(f"Релевантных объявлений в сыром топ-5000 BM25: {present_in_raw}")
print(f"Из них отсеяно ярусным фильтром до топ-50: {lost_by_filter} ({lost_by_filter/present_in_raw:.1%})")


**Разбивка по категориям — находка.** 14 899 из 14 906 валидационных запросов (99.95%) имеют одно и то же значение `search_category = 114`. То есть в этом бенчмарке категория практически не варьируется, весь датасет фактически ограничен одной товарной категорией, и разбивка recall по категориям здесь неинформативна (делить не на что). Практический вывод: категориальный компонент ярусного фильтра в рамках этой задачи почти не работает как дискриминирующий признак — всю полезную фильтрующую нагрузку несёт локация (см. также уточнение в разделе 1).

**Вклад каждого потока — находка, но с методологической оговоркой.** Числа ниже — pooled-метрика (доля от общего числа пар в ground truth, 89 503), а не macro-среднее по запросам, как официальный Recall@50. Эти два способа усреднения закономерно дают разные числа: 57.4% запросов имеют recall=1.0 (как правило, у них мало релевантных объявлений), поэтому macro-среднее (0.7057) сильно тянется вверх лёгкими запросами. Pooled-метрика, наоборот, сильнее взвешена по запросам с большим числом релевантных объявлений — именно они дают основную массу пар и являются самыми сложными.

| | Кол-во пар | Доля от 89 503 |
|---|---|---|
| Только BM25 (в своём топ-50) | 6 708 | 7.5% |
| Только Dense (в своём топ-50) | 7 849 | 8.8% |
| Оба потока (в топ-50 обоих) | 20 660 | 23.1% |
| Ни один из потоков (вне топ-50 обоих) | 54 286 | 60.7% |

Ключевой вывод: объединение индивидуальных топ-50 списков BM25 и Dense покрывает лишь 39.3% пар (7.5+8.8+23.1), тогда как итоговый (macro) Recall@50 после RRF — 70.57%. Разница объясняется тем, что RRF строит финальный топ-50 не из топ-50 каждого потока, а из **топ-200** кандидатов каждого потока: объявление, не попавшее в топ-50 ни одного потока по отдельности, но присутствующее в топ-200 у обоих, может набрать достаточный суммарный RRF-скор для попадания в финальный топ-50. Реальная польза RRF в этом пайплайне — не просто «взять лучшее от каждого потока», а использование более глубокого пула кандидатов (топ-200), недоступного при сравнении только топ-50 списков.


## 7. Инференс на тестовых данных

Повторение пайплайна на `benchmark_items` и `benchmark_queries`:
1. Построение BM25-индекса для тестового корпуса
2. Кодирование через дообученный ruBERT
3. Извлечение кандидатов из обоих потоков с ярусным фильтром
4. Weighted RRF — формирование топ-50
5. Генерация `answer.csv` с валидацией формата

Формат ответа: `query_id,answer` — одна строка на запрос, 50 item_id через пробел.


In [ ]:
# --- Загрузка тестовых данных ---
df_bench_items = pd.read_parquet(bench_items_path)
df_bench_queries = pd.read_parquet(bench_queries_path)

df_bench_queries['query_id'] = df_bench_queries['query_id'].astype(str).str.strip()
df_bench_items['item_id'] = df_bench_items['item_id'].astype(str).str.strip().str.lower()

all_bench_items_list = df_bench_items['item_id'].tolist()
cat_to_items = df_bench_items.groupby('item_category_id')['item_id'].apply(list).to_dict()

print(f"Benchmark: {len(df_bench_items):,} объявлений, {len(df_bench_queries):,} запросов")

# --- BM25 индексация ---
print("Индексация BM25 для тестового корпуса...")
b_title_clean = df_bench_items['item_title_raw'].apply(clean_text)
b_params_clean = df_bench_items['item_infm_params_text'].apply(clean_text)
b_desc_clean = df_bench_items['item_description_raw'].fillna('').str[:600].apply(clean_text)

b_corpus_texts = (
    b_title_clean + " " + b_title_clean + " " + b_title_clean + " " +
    b_params_clean + " " +
    b_desc_clean
).tolist()

b_corpus_ids = df_bench_items['item_id'].to_numpy()
b_loc_tensor = torch.tensor(df_bench_items['item_location_id'].fillna(0).to_numpy(dtype=np.int32), dtype=torch.int32, device=device)
b_cat_tensor = torch.tensor(df_bench_items['item_category_id'].fillna(0).to_numpy(dtype=np.int32), dtype=torch.int32, device=device)

b_corpus_tokens = bm25s.tokenize(b_corpus_texts, stopwords='russian', stemmer=stemmer)
retriever_bench = bm25s.BM25(k1=1.5, b=0.4)
retriever_bench.index(b_corpus_tokens)

b_queries_clean = [clean_text(q) for q in df_bench_queries['search_query']]
b_query_tokens = bm25s.tokenize(b_queries_clean, stopwords='russian', stemmer=stemmer)

del b_corpus_texts, b_corpus_tokens
gc.collect()

# --- Dense кодирование ---
print("Dense-кодирование тестового корпуса и запросов (FP16)...")
b_rich_titles = (b_title_clean + " " + b_params_clean.str[:200]).str.strip().tolist()

with torch.no_grad():
    b_corpus_embs = ft_model.encode(
        b_rich_titles, batch_size=1024, show_progress_bar=True,
        normalize_embeddings=True, convert_to_tensor=True, device=device
    ).to(dtype=emb_dtype)

    b_query_embs = ft_model.encode(
        b_queries_clean, batch_size=1024, show_progress_bar=True,
        normalize_embeddings=True, convert_to_tensor=True, device=device
    ).to(dtype=emb_dtype)

del b_rich_titles
gc.collect()

# --- BM25 кандидаты с ярусным фильтром ---
print("Извлечение BM25-кандидатов...")
b_bm25_cands, _ = retriever_bench.retrieve(b_query_tokens, k=min(5000, len(b_corpus_ids)))
b_bm25_preds = {}
k_bench_cand = b_bm25_cands.shape[1]
rank_penalty_b = torch.arange(k_bench_cand, device=device, dtype=torch.int32)

for i in range(len(df_bench_queries)):
    qid = df_bench_queries.iloc[i]['query_id']
    q_loc = df_bench_queries.iloc[i]['search_location_id']
    q_cat = df_bench_queries.iloc[i]['search_category']
    is_del = df_bench_queries.iloc[i]['search_is_delivery_search']

    cand_idx = torch.tensor(b_bm25_cands[i], dtype=torch.long, device=device)
    c_locs = b_loc_tensor[cand_idx]
    c_cats = b_cat_tensor[cand_idx]

    m_loc = (c_locs == q_loc)
    m_cat = (c_cats == q_cat)
    priority = (m_cat.int() * 20000000 - rank_penalty_b) if is_del == 1 else ((m_loc & m_cat).int() * 20000000 + m_cat.int() * 10000000 - rank_penalty_b)
    top_idx = torch.topk(priority, k=min(200, k_bench_cand)).indices
    b_bm25_preds[qid] = b_corpus_ids[cand_idx[top_idx].cpu().numpy()].tolist()

# --- Dense кандидаты с фильтрацией ---
print("Извлечение Dense-кандидатов...")
b_dense_preds = {}
batch_q_size = 250
num_bench_q = len(df_bench_queries)

for q_start in tqdm(range(0, num_bench_q, batch_q_size), desc="Dense Benchmark"):
    q_end = min(q_start + batch_q_size, num_bench_q)
    batch_q_embs = b_query_embs[q_start:q_end]
    sims_batch = torch.matmul(batch_q_embs, b_corpus_embs.T)

    for local_idx, global_idx in enumerate(range(q_start, q_end)):
        qid = df_bench_queries.iloc[global_idx]['query_id']
        q_loc = df_bench_queries.iloc[global_idx]['search_location_id']
        q_cat = df_bench_queries.iloc[global_idx]['search_category']
        is_del = df_bench_queries.iloc[global_idx]['search_is_delivery_search']

        sims = sims_batch[local_idx]
        s_dt = sims.dtype
        m_loc = (b_loc_tensor == q_loc)
        m_cat = (b_cat_tensor == q_cat)
        priority = (sims + m_cat.to(dtype=s_dt) * 20.0) if is_del == 1 else (sims + (m_loc & m_cat).to(dtype=s_dt) * 20.0 + m_cat.to(dtype=s_dt) * 10.0)
        top_idx = torch.topk(priority, k=200).indices
        b_dense_preds[qid] = b_corpus_ids[top_idx.cpu().numpy()].tolist()

# --- Weighted RRF и формирование ответа ---
print("Weighted RRF и формирование answer.csv...")
sub_rows = []

for i in tqdm(range(len(df_bench_queries)), desc="Submission"):
    qid = str(df_bench_queries.iloc[i]['query_id']).strip()
    q_cat = df_bench_queries.iloc[i]['search_category']

    scores = defaultdict(float)
    bm25_list = b_bm25_preds.get(qid, [])
    dense_list = b_dense_preds.get(qid, [])

    for r, itm in enumerate(bm25_list, 1):
        scores[itm] += opt_wb / (opt_k + r)
    for r, itm in enumerate(dense_list, 1):
        scores[itm] += opt_wd / (opt_k + r)

    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Дедупликация и отбор top-50
    chosen_items = []
    seen = set()
    for itm, _ in sorted_items:
        clean_itm = str(itm).lower().strip()
        if clean_itm not in seen:
            seen.add(clean_itm)
            chosen_items.append(clean_itm)
            if len(chosen_items) == 50:
                break

    # Фоллбэк: добор из той же категории
    if len(chosen_items) < 50 and q_cat in cat_to_items:
        for itm in cat_to_items[q_cat]:
            clean_itm = str(itm).lower().strip()
            if clean_itm not in seen:
                seen.add(clean_itm)
                chosen_items.append(clean_itm)
                if len(chosen_items) == 50:
                    break

    # Фоллбэк: добор из всего каталога
    if len(chosen_items) < 50:
        for itm in all_bench_items_list:
            clean_itm = str(itm).lower().strip()
            if clean_itm not in seen:
                seen.add(clean_itm)
                chosen_items.append(clean_itm)
                if len(chosen_items) == 50:
                    break

    assert len(chosen_items) == 50, f"Не собрано 50 ответов для {qid}"
    sub_rows.append({'query_id': qid, 'answer': ' '.join(chosen_items)})

# --- Сохранение ---
df_sub = pd.DataFrame(sub_rows, columns=['query_id', 'answer'])
df_sub.to_csv('answer.csv', index=False)

# --- Валидация формата ---
print()
print("Валидация формата answer.csv...")
df_check = pd.read_csv('answer.csv', dtype=str)
assert list(df_check.columns) == ['query_id', 'answer'], f"Неверные колонки: {list(df_check.columns)}"
assert len(df_check) == len(df_bench_queries), f"Неверное число строк: {len(df_check)} != {len(df_bench_queries)}"
assert df_check.isna().sum().sum() == 0, "Обнаружены NaN"
for _, row in df_check.iterrows():
    items = row['answer'].strip().split(' ')
    assert len(items) == 50, f"Не 50 объявлений в строке {row['query_id']}"
    assert len(set(items)) == 50, f"Дубликаты в строке {row['query_id']}"
with open('answer.csv', 'r') as f:
    header = f.readline().strip()
assert header == 'query_id,answer', f"Неверный заголовок: {header}"

print("Все проверки пройдены. Файл answer.csv готов к отправке.")
print(f"  Строк: {len(df_check)}")
print(f"  Формат: query_id, 50 item_id через пробел")


## 8. Итоги

### Результаты

| Метод | Recall@50 (валидация) | Recall@50 (тест) |
|---|---|---|
| BM25 v2 (обогащённый текст + стемминг) | 0.6330 | — |
| Bi-Encoder (rubert-tiny2, 4 эпохи) | 0.6580 | — |
| **Weighted RRF (BM25 + Bi-Encoder)** | **0.7057** | **0.7745** |

Разрыв между валидацией (0.7057) и тестовым набором (0.7745, +6.9 п.п.) заметен и заслуживает отдельного комментария. Наиболее вероятная причина — разница в размере каталога: тестовый корпус (`benchmark_items`) содержит 189 212 объявлений, тогда как корпус для валидации, построенный из `train.parquet`, содержит 344 825 уникальных объявлений — почти вдвое больше конкурирующих кандидатов на этапе ранжирования. Меньший каталог при том же k=50 структурно даёт более высокий recall, поэтому прямое сравнение валидационной метрики и метрики на тестовом наборе здесь не совсем корректно, и это стоит держать в уме при интерпретации итогового скора.

### Что сработало

1. **Обогащение текста** (Title x3 + Params + Desc) значительно повысило качество BM25 по сравнению с наивным подходом.
2. **Два потока дополняют друг друга**: BM25 точен при совпадении терминов, Bi-Encoder — при семантическом сходстве (синонимы, перефразировки). См. раздел 6 для количественной оценки вклада каждого потока.
3. **RRF** — простой метод слияния без обучения, который устойчиво даёт прирост над каждым потоком по отдельности: с 0.6330/0.6580 до 0.7057.
4. **Ярусный фильтр** по локации и категории отсекает нерелевантных кандидатов на ранних этапах, при этом не исключая кросс-городские результаты полностью (см. раздел 6).

### Что не сработало

- **CatBoost re-ranking** поверх RRF-кандидатов давал лишь +0.01 Recall@50 при значительных затратах GPU-памяти. Был отклонён как неэффективный.

### Обнаруженные проблемы с данными

1. **Кросс-городская релевантность**: 12.3% правильных ответов находятся в городах, отличных от города поиска (87.7% совпадают и по городу, и по категории). Ярусный фильтр приоритизирует локальные объявления, что создаёт умеренный потолок для recall, но, судя по разделу 6, не является основной причиной нулевого recall у 16.2% запросов — там, вероятно, доминируют другие факторы.
2. **Колонка `item_price`** хранится как строка (object), а не число — требует явного приведения типов при обработке. В текущем пайплайне цена не используется как признак; отмечено как задел на будущее (см. «Возможные улучшения»).

### Возможные улучшения

- Более крупная модель (rubert-base, 180M параметров) при наличии ресурсов.
- Ослабление ярусного фильтра для кросс-городских запросов.
- Cross-Encoder re-ranking для финального переупорядочивания кандидатов.
- Hard negative mining при обучении Bi-Encoder.
